# Complete PDF RAG Pipeline - Using Pre-computed Embeddings

This notebook implements an end-to-end Retrieval-Augmented Generation (RAG) system using **your existing PDF chunks and embeddings**.

**Pipeline Steps:**
1. ✅ **OCR & Chunking**: Already done - using your JSON file
2. ✅ **Embeddings**: Already computed with all-MiniLM-L6-v2
3. **Load Data**: Import JSON into Delta table
4. **Vector Search**: Create index with pre-computed embeddings
5. **RAG Model**: Build retrieval + generation model
6. **Serving Endpoint**: Deploy as real-time API

**Data Source**: `pdf_chunks_with_embeddings.json` (3 PDFs, pre-chunked and embedded)

**Output**: Production-ready RAG endpoint

In [0]:
# Configuration
catalog = "hackathon"
schema = "raw"
json_file_path = "/Workspace/Users/arnav.prasad999918@gmail.com/arthasetu-databricks/output/pdf_chunks_with_embeddings.json"
table_name = f"{catalog}.{schema}.pdf_chunks_embeddings"
vector_search_endpoint_name = "pdf_rag_endpoint"
index_name = f"{catalog}.{schema}.pdf_chunks_index"

print(f"✓ Configuration loaded")
print(f"  - Source JSON: {json_file_path}")
print(f"  - Target table: {table_name}")
print(f"  - Vector index: {index_name}")
print(f"  - VS endpoint: {vector_search_endpoint_name}")

## Step 1: Load Pre-computed Embeddings

Your JSON file contains:
- **source**: PDF filename
- **text**: Chunked text (1000 chars with 200 overlap)
- **embedding**: 384-dim vectors from all-MiniLM-L6-v2

We'll load this directly into a DataFrame and prepare it for Vector Search.

In [0]:
import json
from pyspark.sql.functions import expr, col
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, FloatType

# Define schema for the JSON
schema = StructType([
    StructField("source", StringType(), True),
    StructField("text", StringType(), True),
    StructField("embedding", ArrayType(FloatType()), True)
])

# Load JSON file
df = spark.read.schema(schema).json(json_file_path)

# Add chunk_id (required for Vector Search)
df_with_id = df.withColumn("chunk_id", expr("uuid()")) \
    .select("chunk_id", "source", col("text").alias("chunk"), "embedding")

print(f"✓ Loaded {df_with_id.count()} chunks from JSON")
print(f"✓ Embedding dimension: {len(df_with_id.first()['embedding'])}")
print(f"\nSample data:")
display(df_with_id.limit(5))

## Step 2: Write to Delta Table

Vector Search requires:
1. Data stored in a Delta table
2. A primary key column (chunk_id) ✓
3. An embedding vector column ✓
4. Change Data Feed enabled for incremental updates

We'll write the loaded data to Delta and enable CDC.

In [0]:
%pip install databricks-vectorsearch mlflow databricks-sdk --upgrade --quiet
dbutils.library.restartPython()

In [0]:
# Write embeddings to Delta table (after Python restart)
# Re-define config variables
catalog = "hackathon"
schema = "raw"
json_file_path = "/Workspace/Users/arnav.prasad999918@gmail.com/arthasetu-databricks/output/pdf_chunks_with_embeddings.json"
table_name = f"{catalog}.{schema}.pdf_chunks_embeddings"

# Re-import libraries
from pyspark.sql.functions import expr, col
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, FloatType

# Define schema
data_schema = StructType([
    StructField("source", StringType(), True),
    StructField("text", StringType(), True),
    StructField("embedding", ArrayType(FloatType()), True)
])

# Reload JSON and add chunk_id
df = spark.read.schema(data_schema).json(json_file_path)
df_final = df.withColumn("chunk_id", expr("uuid()")) \
    .select("chunk_id", "source", col("text").alias("chunk"), "embedding")

# Write to Delta
df_final.write.format("delta").mode("overwrite").saveAsTable(table_name)

# Enable Change Data Feed
spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"✓ Saved {df_final.count()} chunks to {table_name}")
print(f"✓ Enabled Change Data Feed")
print(f"\nSample data:")
display(spark.table(table_name).limit(5))

## Step 3: Create Vector Search Endpoint

A Vector Search endpoint is the compute infrastructure that hosts vector indexes. We'll create a STANDARD endpoint that can host multiple indexes.

In [0]:
# Create Vector Search endpoint
# Config
catalog = "hackathon"
schema = "raw"
table_name = f"{catalog}.{schema}.pdf_chunks_embeddings"
vector_search_endpoint_name = "pdf_rag_endpoint"
index_name = f"{catalog}.{schema}.pdf_chunks_index"

from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

# Create endpoint if it doesn't exist
try:
    endpoint = vsc.get_endpoint(vector_search_endpoint_name)
    print(f"✓ Endpoint '{vector_search_endpoint_name}' already exists")
    print(f"  Status: {endpoint.get('endpoint_status', {}).get('state', 'UNKNOWN')}")
except:
    # Endpoint doesn't exist, create it
    print(f"Creating endpoint '{vector_search_endpoint_name}'...")
    vsc.create_endpoint(
        name=vector_search_endpoint_name,
        endpoint_type="STANDARD"
    )
    print(f"✓ Created endpoint '{vector_search_endpoint_name}'")
    print("  Note: Endpoint provisioning takes 5-10 minutes")

## Step 4: Create Vector Index with Pre-computed Embeddings

We'll create a vector index that:
- **Uses your pre-computed embeddings** (no re-embedding needed!)
- **Syncs with Delta table** for automatic updates
- **Supports similarity search** for RAG retrieval

The index will use the existing 384-dim all-MiniLM-L6-v2 embeddings.

In [0]:
import time

# Wait for endpoint to be online
print(f"Checking endpoint status...")
for i in range(30):
    try:
        endpoint_status = vsc.get_endpoint(vector_search_endpoint_name)
        state = endpoint_status.get('endpoint_status', {}).get('state', 'UNKNOWN')
        if state == 'ONLINE':
            print(f"✓ Endpoint is ONLINE")
            break
        else:
            print(f"  Endpoint state: {state}, waiting...")
            time.sleep(20)
    except Exception as e:
        print(f"  Waiting for endpoint... ({i*20}s)")
        time.sleep(20)

# Create vector index with pre-computed embeddings
try:
    index = vsc.create_delta_sync_index(
        endpoint_name=vector_search_endpoint_name,
        index_name=index_name,
        source_table_name=table_name,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_vector_column="embedding",  # Use pre-computed embeddings!
        embedding_dimension=384  # all-MiniLM-L6-v2 dimension
    )
    print(f"✓ Created index '{index_name}'")
    print(f"  Using pre-computed embeddings (384 dims)")
    print(f"  Primary key: chunk_id")
    print(f"  Embedding column: embedding")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"✓ Index '{index_name}' already exists, syncing...")
        index = vsc.get_index(endpoint_name=vector_search_endpoint_name, index_name=index_name)
        index.sync()
        print(f"✓ Index sync triggered")
    else:
        raise e

In [0]:
import time

print("Waiting for index to be ready...")
print("This should be fast since embeddings are pre-computed!\n")

for i in range(60):
    try:
        status = vsc.get_index(endpoint_name=vector_search_endpoint_name, index_name=index_name).describe()
        index_status = status.get('status', {})
        
        if index_status.get('ready', False):
            print(f"\n✓ Index is READY!")
            print(f"  Indexed rows: {status.get('num_indexed_rows', 'unknown')}")
            break
        else:
            message = index_status.get('message', 'Indexing...')
            if i % 3 == 0:  # Print every 30 seconds
                print(f"  [{i*10}s] {message}")
    except Exception as e:
        print(f"  Checking status... ({i*10}s)")
    
    time.sleep(10)

print("\n✓ Vector index ready for similarity search!")

## Step 5: Test Vector Search

Let's verify the vector index works by performing a similarity search. This retrieves the most relevant chunks based on semantic similarity.

In [0]:
# Get the index
index = vsc.get_index(endpoint_name=vector_search_endpoint_name, index_name=index_name)

# Test search
test_query = "What is this document about?"
results = index.similarity_search(
    query_text=test_query,
    columns=["chunk_id", "source_doc", "chunk"],
    num_results=3
)

print(f"Query: '{test_query}'")
print(f"\n=== Top 3 Similar Chunks ===\n")

for idx, row in enumerate(results.get('result', {}).get('data_array', []), 1):
    chunk_id = row[0]
    source_doc = row[1]
    chunk_text = row[2]
    score = row[-1] if len(row) > 3 else "N/A"
    
    print(f"Result {idx}:")
    print(f"  Source: {source_doc}")
    print(f"  Chunk ID: {chunk_id}")
    print(f"  Score: {score}")
    print(f"  Text: {chunk_text[:300]}...")
    print()

## Step 6: Deploy RAG Model to Serving Endpoint

Now we'll create a custom RAG model that:
1. **Retrieves** relevant chunks using vector similarity search
2. **Augments** the query with retrieved context
3. **Generates** answers using Databricks Foundation Model API (Llama 3.1 70B)

The model will be logged with MLflow and deployed as a serving endpoint.

In [0]:
import mlflow
from mlflow.models import infer_signature
import pandas as pd

class RAGModel(mlflow.pyfunc.PythonModel):
    """Custom RAG model that retrieves context and generates answers"""
    
    def __init__(self, vector_search_endpoint, index_name):
        self.vector_search_endpoint = vector_search_endpoint
        self.index_name = index_name
        
    def load_context(self, context):
        """Load dependencies when model is loaded"""
        from databricks.vector_search.client import VectorSearchClient
        self.vsc = VectorSearchClient()
        self.index = self.vsc.get_index(
            endpoint_name=self.vector_search_endpoint,
            index_name=self.index_name
        )
    
    def predict(self, context, model_input):
        """Generate answers based on retrieved context"""
        from databricks.sdk import WorkspaceClient
        
        results = []
        questions = model_input["question"].tolist() if hasattr(model_input["question"], "tolist") else [model_input["question"][0]]
        
        for question in questions:
            # Step 1: Retrieve relevant chunks
            search_results = self.index.similarity_search(
                query_text=question,
                columns=["chunk", "source_doc"],
                num_results=3
            )
            
            # Step 2: Extract context
            context_chunks = []
            sources = []
            for row in search_results.get('result', {}).get('data_array', []):
                chunk_text = row[0]
                source_doc = row[1]
                context_chunks.append(f"[Source: {source_doc}]\n{chunk_text}")
                sources.append(source_doc)
            
            context_text = "\n\n---\n\n".join(context_chunks)
            
            # Step 3: Generate answer using Foundation Model API
            w = WorkspaceClient()
            prompt = f"""Based on the following context from PDF documents, answer the question. If the answer is not in the context, say "I don't have enough information to answer this question."

Context:
{context_text}

Question: {question}

Answer:"""
            
            response = w.serving_endpoints.query(
                name="databricks-meta-llama-3-1-70b-instruct",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=500
            )
            
            answer = response.choices[0].message.content
            
            results.append({
                "question": question,
                "answer": answer,
                "sources": list(set(sources))  # Unique sources
            })
        
        return pd.DataFrame(results)

# Log model to MLflow
mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="pdf_rag_model") as run:
    model = RAGModel(
        vector_search_endpoint=vector_search_endpoint_name,
        index_name=index_name
    )
    
    # Create signature
    input_example = pd.DataFrame({"question": ["What is this document about?"]})
    output_example = pd.DataFrame({
        "question": ["test"],
        "answer": ["test answer"],
        "sources": [["doc1.pdf"]]
    })
    signature = infer_signature(input_example, output_example)
    
    mlflow.pyfunc.log_model(
        artifact_path="rag_model",
        python_model=model,
        signature=signature,
        input_example=input_example,
        pip_requirements=[
            "databricks-vectorsearch",
            "databricks-sdk",
            "mlflow"
        ]
    )
    
    model_uri = f"runs:/{run.info.run_id}/rag_model"
    print(f"✓ Model logged to MLflow")
    print(f"  Run ID: {run.info.run_id}")
    print(f"  Model URI: {model_uri}")

## Step 7: Register and Deploy Model to Serving Endpoint

We'll register the model in Unity Catalog and deploy it as a real-time serving endpoint with:
- **Auto-scaling**: Scales to zero when not in use
- **Small workload size**: Cost-effective for development/testing
- **REST API**: Query via HTTP from any application

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServedEntityInput, EndpointCoreConfigInput

# Register model in Unity Catalog
model_name = f"{catalog}.{schema}.pdf_rag_model"
registered_model = mlflow.register_model(model_uri, model_name)
print(f"✓ Registered model: {model_name}")
print(f"  Version: {registered_model.version}")

# Deploy to serving endpoint
w = WorkspaceClient()
serving_endpoint_name = "pdf-rag-endpoint"

try:
    # Create endpoint
    w.serving_endpoints.create(
        name=serving_endpoint_name,
        config=EndpointCoreConfigInput(
            served_entities=[
                ServedEntityInput(
                    entity_name=model_name,
                    entity_version=registered_model.version,
                    scale_to_zero_enabled=True,
                    workload_size="Small"
                )
            ]
        )
    )
    print(f"\n✓ Created serving endpoint: {serving_endpoint_name}")
except Exception as e:
    if "already exists" in str(e).lower():
        # Update existing endpoint
        w.serving_endpoints.update_config(
            name=serving_endpoint_name,
            served_entities=[
                ServedEntityInput(
                    entity_name=model_name,
                    entity_version=registered_model.version,
                    scale_to_zero_enabled=True,
                    workload_size="Small"
                )
            ]
        )
        print(f"\n✓ Updated serving endpoint: {serving_endpoint_name}")
    else:
        raise e

print(f"  Model: {model_name} v{registered_model.version}")
print(f"  Workload: Small (scale to zero enabled)")
print(f"\n✓ Endpoint URL: {w.config.host}/ml/endpoints/{serving_endpoint_name}")

## Step 8: Test the RAG Endpoint

Once the endpoint is ready, we'll send test queries to verify the complete RAG pipeline works end-to-end.

In [0]:
import time

w = WorkspaceClient()

# Wait for endpoint to be ready
print("Waiting for endpoint to be ready...")
print("This may take 10-15 minutes for initial deployment.\n")

for i in range(90):
    try:
        endpoint_status = w.serving_endpoints.get(serving_endpoint_name)
        state = endpoint_status.state.ready
        config_state = endpoint_status.state.config_update
        
        if state == "READY" and config_state == "NOT_UPDATING":
            print(f"\n✓ Endpoint is READY!\n")
            break
        else:
            if i % 6 == 0:  # Print every minute
                print(f"  [{i*10}s] State: {state}, Config: {config_state}")
    except Exception as e:
        if i % 6 == 0:
            print(f"  [{i*10}s] Waiting for endpoint...")
    
    time.sleep(10)

# Test the endpoint
test_questions = [
    "What are the main topics covered in these documents?",
    "Can you summarize the key points from the PDFs?"
]

for question in test_questions:
    print(f"\n{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}\n")
    
    try:
        response = w.serving_endpoints.query(
            name=serving_endpoint_name,
            dataframe_records=[{"question": question}]
        )
        
        prediction = response.predictions[0]
        print(f"Answer:\n{prediction['answer']}\n")
        print(f"Sources: {', '.join(prediction['sources'])}")
    except Exception as e:
        print(f"Error: {e}")
        print("Make sure the endpoint is fully ready and try again.")

## 🎉 RAG Pipeline Complete!

### Resources Created:

**Data Assets:**
- `{catalog}.{schema}.pdf_raw_parsed` - Parsed PDF content with OCR
- `{catalog}.{schema}.pdf_chunks_embeddings` - Chunked text with embeddings

**Vector Search:**
- Endpoint: `{vector_search_endpoint_name}`
- Index: `{catalog}.{schema}.pdf_chunks_index`
- Embedding Model: `databricks-bge-large-en`

**Model & Serving:**
- Model: `{catalog}.{schema}.pdf_rag_model`
- Serving Endpoint: `pdf-rag-endpoint`
- LLM: `databricks-meta-llama-3-1-70b-instruct`

### How to Use:

**Query via Python:**
```python
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

response = w.serving_endpoints.query(
    name="pdf-rag-endpoint",
    dataframe_records=[{"question": "Your question here?"}]
)
print(response.predictions[0]['answer'])
```

**Query via REST API:**
```bash
curl -X POST https://<workspace-url>/serving-endpoints/pdf-rag-endpoint/invocations \
  -H "Authorization: Bearer <token>" \
  -H "Content-Type: application/json" \
  -d '{"dataframe_records": [{"question": "Your question?"}]}'
```

### Next Steps:
- Add more PDFs to `/Volumes/hackathon/default/raw/` and re-run cells 4-10
- Tune chunk size/overlap in the chunking function
- Adjust number of retrieved chunks (num_results) in RAG model
- Monitor endpoint usage in Databricks UI